<a href="https://colab.research.google.com/github/MBeckyAlek/Formative3_ML_C1/blob/alec-branch/Formative3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re

In [ ]:
df = pd.read_csv('GaltonFamilies.csv')
df = df[["father", "mother", "childHeight"]]
df.head()

,father,mother,childHeight
0,78.5,67.0,73.2
1,78.5,67.0,69.2
2,78.5,67.0,69.0
3,78.5,67.0,69.0
4,75.5,66.5,73.5


In [ ]:
fathers = df['father'].values
children = df['childHeight'].values

heights = np.concatenate([fathers,children])

np.random.seed(42)
np.random.shuffle(heights)

print(f"Combined dataset created!")
print(f"Total heights records : {len(heights)}")
print(f"sample heights : {heights[:5]}")

Combined dataset created!
Total heights records : 1868
sample heights : [70.2 62.  64.  70.  69. ]


In [ ]:
# Helper function: Gaussian Probability Density Function
def gaussian_pdf(x, mu, sigma2):
    sigma = np.sqrt(sigma2)
    coefficient = 1.0 / (np.sqrt(2 * np.pi) * sigma)
    exponent = np.exp(-0.5 * ((x - mu) ** 2) / sigma2)
    return coefficient * exponent

# 1. Initialization (Iteration 0)
np.random.seed(42)
mu1 = np.min(heights) + 5.0
mu2 = np.max(heights) - 5.0
sigma1_2 = 100.0
sigma2_2 = 100.0
pi1 = 0.5
pi2 = 0.5

tracking_table = []
data_heights = heights.astype(float)
num_iterations = 10  # We run 5 to get 0, 1, 2 comfortably

for iteration in range(num_iterations):
    # --- E-STEP ---
    pdf1 = gaussian_pdf(data_heights, mu1, sigma1_2)
    pdf2 = gaussian_pdf(data_heights, mu2, sigma2_2)

    weighted_pdf1 = pi1 * pdf1
    weighted_pdf2 = pi2 * pdf2
    total_likelihood = weighted_pdf1 + weighted_pdf2

    resp1 = weighted_pdf1 / total_likelihood
    resp2 = weighted_pdf2 / total_likelihood

    log_likelihood = np.sum(np.log(total_likelihood + 1e-10))

    # Save metrics for tracking table
    tracking_table.append({
        "Iteration": iteration,
        "μ1 (Children)": round(mu1, 4),
        "μ2 (Pros)": round(mu2, 4),
        "σ1²": round(sigma1_2, 4),
        "σ2²": round(sigma2_2, 4),
        "π1": round(pi1, 4),
        "π2": round(pi2, 4),
        "Log-Likelihood": round(log_likelihood, 4)
    })

    # --- M-STEP ---
    N1 = np.sum(resp1)
    N2 = np.sum(resp2)
    N = len(data_heights)

    mu1 = np.sum(resp1 * data_heights) / N1
    mu2 = np.sum(resp2 * data_heights) / N2

    sigma1_2 = np.sum(resp1 * (data_heights - mu1) ** 2) / N1
    sigma2_2 = np.sum(resp2 * (data_heights - mu2) ** 2) / N2

    pi1 = N1 / N
    pi2 = N2 / N

# Print out the tracking table
tracking_df = pd.DataFrame(tracking_table)
print("--- OPTIMIZATION TRACKING TABLE ---")
display(tracking_df)

--- OPTIMIZATION TRACKING TABLE ---


,Iteration,μ1 (Children),μ2 (Pros),σ1²,σ2²,π1,π2,Log-Likelihood
0,0,61.0000,74.0000,100.0000,100.0000,0.5000,0.5000,-6473.6808
1,1,67.2670,68.6349,11.0811,9.9461,0.4849,0.5151,-4884.5502
2,2,67.2466,68.6532,11.5514,9.4526,0.4846,0.5154,-4882.5651
3,3,67.2138,68.6832,11.9329,9.0079,0.4843,0.5157,-4880.8678
4,4,67.1726,68.7212,12.2390,8.6065,0.4841,0.5159,-4879.3532
5,5,67.1261,68.7645,12.4779,8.2447,0.4840,0.5160,-4877.9767
6,6,67.0765,68.8112,12.6566,7.9194,0.4841,0.5159,-4876.7170
7,7,67.0258,68.8598,12.7814,7.6275,0.4844,0.5156,-4875.5615
8,8,66.9754,68.9095,12.8588,7.3657,0.4849,0.5151,-4874.5001
9,9,66.9264,68.9592,12.8956,7.1306,0.4859,0.5141,-4873.5236


### Part 2: Bayesian Probability

0. Load the dataset and count the number of reviews

In [ ]:
import csv

# Read the dataset robustly handling potential quoting issues
data = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")
total_reviews = len(data)
print(data.head())
print("reviews:", total_reviews)
print(data["sentiment"].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
reviews: 6270
sentiment
negative    3159
positive    3111
Name: count, dtype: int64


1. Key Word Selection

In [ ]:

# Define positive and negative keywords

positive_words = ["fantastic", "wonderful", "stunning", "best"]
negative_words = ["bad", "boring", "awful", "worst"]

print("Positive keywords:", positive_words)
print("Negative keywords:", negative_words)


Positive keywords: ['fantastic', 'wonderful', 'stunning', 'best']
Negative keywords: ['bad', 'boring', 'awful', 'worst']


2. Choice of Conditional Probability:

We calculate the ratio of how many times the selected key words appeared on each sentiment lqabeld as postive agains negtaibve and vice versa, this to help us with the choice of the conditional probability and in this case we went with P(Positive)

In [ ]:

# Store the counts for each keyword
keyword_counts = {}

# Combine both keyword lists
all_words = positive_words + negative_words

for word in all_words:
    # Find reviews containing this word
    contains_word = data["review"].str.contains(
        r"\b" + re.escape(word) + r"\b",
        case=False,
        na=False
    )

    # Count reviews containing the word for each sentiment
    positive_count = ((data["sentiment"] == "positive") & contains_word).sum()
    negative_count = ((data["sentiment"] == "negative") & contains_word).sum()

    keyword_counts[word] = {
        "positive": positive_count,
        "negative": negative_count
    }

# Display the results
for word, counts in keyword_counts.items():
    print(
        f"{word}: Positive = {counts['positive']}, "
        f"Negative = {counts['negative']}"
    )


fantastic: Positive = 138, Negative = 29
wonderful: Positive = 317, Negative = 66
stunning: Positive = 72, Negative = 20
best: Positive = 768, Negative = 423
bad: Positive = 352, Negative = 1127
boring: Positive = 69, Negative = 320
awful: Positive = 35, Negative = 338
worst: Positive = 53, Negative = 481


3. Computation and Presentation

For each chosen keyword, create a small table or markdown block showing the following probabilities:


In [ ]:
keywords = positive_words

# Prepare the review text and sentiment columns
review_text = data["review"].fillna("").str.lower()
sentiment = data["sentiment"].str.lower()

# 1. Prior
total_reviews = len(data)
positive_total = (sentiment == "positive").sum()
p_positive = positive_total / total_reviews

results = []

for keyword in keywords:

    # Find reviews containing the keyword
    contains_keyword = review_text.str.contains(
        r"\b" + re.escape(keyword) + r"\b",
        regex=True,
        na=False
    )

    all_with_keyword = contains_keyword.sum()

    positive_with_keyword = (
        (sentiment == "positive") & contains_keyword
    ).sum()

    # 2. Likelihood
    likelihood = positive_with_keyword / positive_total

    # 3. Marginal
    marginal = all_with_keyword / total_reviews

    # 4. Posterior
    if marginal > 0:
        posterior = (likelihood * p_positive) / marginal
    else:
        posterior = None

    results.append({
        "Keyword": keyword,
        "Prior P(Positive)": p_positive,
        "Likelihood P(keyword|Positive)": likelihood,
        "Marginal P(keyword)": marginal,
        "Posterior P(Positive|keyword)": posterior
    })

# Display the results
results_df = pd.DataFrame(results)
display(results_df.round(4))


,Keyword,Prior P(Positive),Likelihood P(keyword|Positive),Marginal P(keyword),Posterior P(Positive|keyword)
0,fantastic,0.4962,0.0444,0.0266,0.8263
1,wonderful,0.4962,0.1019,0.0611,0.8277
2,stunning,0.4962,0.0231,0.0147,0.7826
3,best,0.4962,0.2469,0.1900,0.6448


# Part 4: Gradient descent

In [ ]:
rng = np.random.default_rng(seed=42)

w = np.array([-1, 2]).reshape(-1, 1)
# b = np.array([1]).reshape(-1, 1)
b = np.float64(1)
alpha = 0.01

data = pd.DataFrame({
    "y": [5, 6],
    "x_i1": [1, 4],
    "x_i2": [3, 10]
})

data, w.shape, b.shape

(   y  x_i1  x_i2
 0  5     1     3
 1  6     4    10,
 (2, 1),
 ())

In [ ]:
X = data.copy().drop(columns=['y']).to_numpy()
y = data.copy()['y'].to_numpy().reshape(-1, 1)

X.shape, y.shape

((2, 2), (2, 1))

In [ ]:
def forward_propagation(weights, bias, X):
    return X @ weights + bias

def mean_squared_error(y_actual, y_pred):
    return np.mean((y_actual - y_pred) ** 2)

def mse_derivative(y_actual, y_pred):
    n = len(y_actual)
    return (2 / n) * (y_pred - y_actual)

def update_params(w, b, dw, db, alpha):
    w = w - (alpha * dw)
    b = b - (alpha * db)
    return w, b

In [ ]:
for i in range(3):
    y_pred = forward_propagation(w, b, X)
    mse = mean_squared_error(y, y_pred)
    gradient = mse_derivative(y, y_pred)
    dw = X.T @ gradient
    db = np.sum(gradient)
    w, b = update_params(w, b, dw, db, alpha)

    print(f"iteration: {i}")
    print("MSE:", mse)
    print("w:", w)
    print("b:", b)
    print("----------------")

iteration: 0
MSE: 61.0
w: [[-1.45]
 [ 0.87]]
b: 0.88
----------------
iteration: 1
MSE: 6.845000000000002
w: [[-1.3316]
 [ 1.1808]]
b: 0.9318
----------------
iteration: 2
MSE: 2.72381716
w: [[-1.369562]
 [ 1.095182]]
b: 0.93624
----------------
